In [1]:
import sys
import git
import pathlib

# Set up the PROJ_ROOT variable
PROJ_ROOT_PATH = pathlib.Path(git.Repo('.', search_parent_directories=True).working_tree_dir)
PROJ_ROOT =  str(PROJ_ROOT_PATH)
if PROJ_ROOT not in sys.path:
    sys.path.append(PROJ_ROOT)

# Explicitly add the current notebook's directory
CURRENT_DIR = str(pathlib.Path().absolute())
if CURRENT_DIR not in sys.path:
    sys.path.insert(0, CURRENT_DIR)

In [2]:
import numpy as np
from scipy.integrate import quad
import math
import matplotlib.pyplot as plt
from library.utils import fontstyle
title_font, axis_label_font, tick_label_font, legend_font, text_font = fontstyle

In [3]:
from library.fridges import TEMP_STAGES
from library.cables import Ag, NbTi

We first determine the *linear* thermal conductivity for Ag given the values in Pg. 18 of [the Delft CrioFlex datasheet](http://photonteck.com/uploads/20241109/b3531c2c55a53a8e034ba1e998f3f7ee.pdf)

In [4]:
Ag_cable = Ag("Ag")
Ag_cable.thermal_conductivity # W-cm/K

{'RT': None,
 '50K': 0.00022222222222222226,
 '4K': 0.0003304347826086957,
 'Still': 2.9333333333333333e-05,
 'CP': 4.444444444444444e-06,
 'MXC': 1.1111111111111108e-06}

From Pg. 8 of [the Delft CrioFlex datasheet](http://photonteck.com/uploads/20241109/b3531c2c55a53a8e034ba1e998f3f7ee.pdf), we can easily see that the linear thermal conductivities of NbTi stripline for Still, CP and MXC is one-tenth that of Ag.

# Note: The following is simply for sanity check.

Using the inferred linear thermal conductivity values, we determine the lengths of the wires in Pg. 8 of [the Delft CrioFlex datasheet](http://photonteck.com/uploads/20241109/b3531c2c55a53a8e034ba1e998f3f7ee.pdf) for the given PHL and temperatures of Ag stripline.

In [5]:
# http://photonteck.com/uploads/20241109/b3531c2c55a53a8e034ba1e998f3f7ee.pdf (Pg. 8)

PHL_Ag = {
    'RT': None,
    '50K': 2.7E-3,
    '4K': 0.8E-3,
    'Still': 5.4E-6,
    'CP': 290E-9,
    'MXC': 5.9E-9
}

operating_temp = {
    'RT': 300,
    '50K': 50,
    '4K': 4,
    'Still': 0.6,
    'CP': 100E-3,
    'MXC': 10E-3
}

cable_length = {}
for temp_stage in TEMP_STAGES[1:]:
    
    # get operating temperatures
    current_temp_stage_idx = TEMP_STAGES.index(temp_stage)
    if current_temp_stage_idx > 0:
        prev_temp_stage = TEMP_STAGES[current_temp_stage_idx - 1]
    T_lo = operating_temp[temp_stage]
    T_hi = operating_temp[prev_temp_stage]
    
    cable_length[temp_stage] = (Ag_cable.thermal_conductivity[temp_stage] * (T_hi-T_lo))  / PHL_Ag[temp_stage]


In [6]:
cable_length

{'50K': 20.5761316872428,
 '4K': 19.0,
 'Still': 18.469135802469133,
 'CP': 7.662835249042145,
 'MXC': 16.94915254237288}

Using the inferred cable lenghts, and the reported PHL values and temperatures for NbTi, we can calculate the linear thermal conductivity for NbTi.

In [7]:
# http://photonteck.com/uploads/20241109/b3531c2c55a53a8e034ba1e998f3f7ee.pdf (Pg. 8)

PHL_NbTi = {
    'RT': None,
    '50K': None,
    '4K': None,
    'Still': 540E-9,
    'CP': 29E-9,
    'MXC': 590E-12
}

In [8]:
NbTi_thermal_conductivity = {
    'RT': None,
    '50K': None,
    '4K': None,
    'Still': None,
    'CP': None,
    'MXC': None
}

for temp_stage in TEMP_STAGES[3:]:
    # get operating temperatures
    current_temp_stage_idx = TEMP_STAGES.index(temp_stage)
    if current_temp_stage_idx > 0:
        prev_temp_stage = TEMP_STAGES[current_temp_stage_idx - 1]
    T_lo = operating_temp[temp_stage]
    T_hi = operating_temp[prev_temp_stage]
    
    NbTi_thermal_conductivity[temp_stage] = (PHL_NbTi[temp_stage] * cable_length[temp_stage] ) / (T_hi-T_lo)

In [9]:
Ag_cable.thermal_conductivity

{'RT': None,
 '50K': 0.00022222222222222226,
 '4K': 0.0003304347826086957,
 'Still': 2.9333333333333333e-05,
 'CP': 4.444444444444444e-06,
 'MXC': 1.1111111111111108e-06}

In [10]:
NbTi_thermal_conductivity

{'RT': None,
 '50K': None,
 '4K': None,
 'Still': 2.933333333333333e-06,
 'CP': 4.4444444444444444e-07,
 'MXC': 1.1111111111111108e-07}

Verifying with the values output by `NbTi` class in `cables.py`

In [11]:
NbTi_cable = NbTi("NbTi")
NbTi_cable.thermal_conductivity # W-cm/K

{'RT': None,
 '50K': None,
 '4K': None,
 'Still': 2.9333333333333333e-06,
 'CP': 4.4444444444444444e-07,
 'MXC': 1.1111111111111108e-07}

We observe that NbTi linear thermal conductivity is one-tenth of Ag.

# Double-checking

In [12]:
PHL_NbTi = {
    'RT': None,
    '50K': None,
    '4K': None,
    'Still': None,
    'CP': None,
    'MXC': None
}

for temp_stage in TEMP_STAGES[3:]:
    # get operating temperatures
    current_temp_stage_idx = TEMP_STAGES.index(temp_stage)
    if current_temp_stage_idx > 0:
        prev_temp_stage = TEMP_STAGES[current_temp_stage_idx - 1]
    T_lo = operating_temp[temp_stage]
    T_hi = operating_temp[prev_temp_stage]
    
    PHL_NbTi[temp_stage] = (NbTi_thermal_conductivity[temp_stage] * (T_hi-T_lo)) / cable_length[temp_stage] 

In [13]:
PHL_NbTi

{'RT': None,
 '50K': None,
 '4K': None,
 'Still': 5.4e-07,
 'CP': 2.9e-08,
 'MXC': 5.9e-10}

In [14]:
PHL_Ag_2 = {
    'RT': None,
    '50K': None,
    '4K': None,
    'Still': None,
    'CP': None,
    'MXC': None
}

for temp_stage in TEMP_STAGES[1:]:
    # get operating temperatures
    current_temp_stage_idx = TEMP_STAGES.index(temp_stage)
    if current_temp_stage_idx > 0:
        prev_temp_stage = TEMP_STAGES[current_temp_stage_idx - 1]
    T_lo = operating_temp[temp_stage]
    T_hi = operating_temp[prev_temp_stage]
    
    PHL_Ag_2[temp_stage] = (Ag_cable.thermal_conductivity[temp_stage] * (T_hi-T_lo)) / cable_length[temp_stage] 

In [15]:
PHL_Ag_2

{'RT': None,
 '50K': 0.0027,
 '4K': 0.0008,
 'Still': 5.4e-06,
 'CP': 2.9e-07,
 'MXC': 5.9e-09}